# Couette T1 Stress-Relaxation Production Run

Drives a bidisperse 2D emulsion in an annular Couette cell at random close packing
(φ ≈ 0.84) and captures the state at every 1250 steps — positions, velocities, and
the full neighbour topology — for offline post-processing of T1 (neighbour-swap)
events and their associated stress-relaxation signatures.

**Setup spec** is documented in `papers/T1_couette_setup/main.pdf`.  
**Geometry recipe** (how `R_inner`, `R_outer`, `P` relate at fixed φ) is in
`docs/couette_design.md`.

## High-level flow

1. **Clone repo + install** the C++ extension.
2. **Mount Google Drive** so checkpoints/results survive Colab session restarts
   (24-hr session limit on Pro+).
3. **Set parameters** — system size, packing fraction, drive rate, segment counts.
4. **Build the system + particles**, swell to target φ, equilibrate.
5. **Identify driven inner ring + pinned outer ring** of frozen droplets.
6. **Drive in 12 segments of 50 000 steps each**, saving state + neighbours
   every 1250 steps.  Each segment writes its own `segment_NNN.npz` to Drive
   atomically; a script crash or session timeout loses at most one segment.
7. **Final concatenation + cumulative movie**.

## When you'd want to adjust

- **For a runtime test only:** set `N_SEGMENTS = 2` (so total run is ~30 min instead
  of ~5 hr on RTX 3080, ~10 min vs ~1 hr on A100).
- **For different physics:** see the parameter cell — every knob is documented inline.
- **To resume after a Colab disconnect:** just re-run the whole notebook.  The
  resume-detection logic auto-finds the latest segment in Drive and continues.


## 1. Setup — clone repo and build the C++ extension

Colab already has `numpy`, `scipy`, `matplotlib`, `imageio`, `tensorflow` (GPU-tuned),
and system `ffmpeg`.  We only need to clone this repo and `pip install .` it,
which builds the `_candidacy_cpp` C++ extension via `scikit-build-core`.


In [ ]:
import os, sys
REPO_URL = 'https://github.com/KD-physics/Squishy-Particle-Simulator.git'
if not os.path.exists('Squishy-Particle-Simulator'):
    !git clone {REPO_URL}
%cd Squishy-Particle-Simulator/python_v2
!pip install -q .
print('Setup complete.')

import tensorflow as tf
print(f'TF {tf.__version__}, GPU: {tf.config.list_physical_devices("GPU")}')


## 2. Mount Google Drive for persistent outputs

All checkpoints, segment data, previews, and the final movie write to a folder
in your Drive.  This means the run survives a Colab session disconnect — just
re-run the notebook and it picks up from the last completed segment.

**Output structure (in your Drive):**
```
EPD/test_C_couette_T1/
├── state_post_swell.npz          ← post-swell checkpoint (skip swell on resume)
├── state_drive.npz               ← latest drive checkpoint
├── meta.npz                      ← segment counter, ring assignments
├── segment_001.npz … segment_012.npz   ← per-segment data
├── preview_step050k.mp4 …                ← per-segment preview movies
├── drive_data.npz                ← final concatenation of all segments
└── full.mp4                      ← final cumulative movie (480 frames)
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
OUT_DIR = pathlib.Path('/content/drive/MyDrive/EPD/test_C_couette_T1')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Outputs → {OUT_DIR}')


## 3. Parameters — every knob with a one-line explanation

| Parameter | Value | What it controls |
|---|---|---|
| `P_TARGET` | 635 | Number of droplets.  Set by the geometry recipe (see `docs/couette_design.md`); changing it requires re-sizing the cell. |
| `N_NODES` | 60 | Perimeter nodes per droplet.  Higher = smoother shape, slower per step.  60 is the calibrated default. |
| `PHI_TARGET` | 0.84 | Target packing fraction in the annulus.  RCP-2D bidisperse ≈ 0.86; we leave a little safety margin. |
| `OH_DRAG` | 0.15 | Ohnesorge number → per-node Stokes drag.  Required during equilibration to drain kinetic energy. |
| `N_RELAX` | 15 000 | Steps of relax with drag (no drive) after swell — drains the swell-induced KE. |
| `SEG_STEPS` | 50 000 | Steps per segment.  Each segment ends with a checkpoint write. |
| `N_SEGMENTS` | 12 | Total: 600 000 driven steps.  **For a runtime test, drop to 2** (~10 min on A100). |
| `SAMPLE_DRIVE` | 1250 | Capture cadence (40 frames per segment, 480 total).  Smaller = denser data, larger files. |
| `OMEGA` | 0.002 | Inner-ring angular velocity (in capillary units).  Quasi-static — much smaller than damping rate. |
| `FPS` | 12 | Frame rate of preview/final movies. |
| `E_CAND` | 64 | Candidacy matrix width.  64 needed at φ ≈ 0.86; 32 sufficient for φ ≤ 0.5. |
| `DT_FACTOR_DRIVE` | 0.8 | dt scale during driving (0.8 × CFL bound).  Empirically: 1.5 trips CRITICAL at this φ; 0.8 stays in advisory range. |
| `POLY_DIST` | bimodal 50/50 | Bidisperse mixture R₀ ∈ {0.8, 1.2}.  Frustrates crystallisation at high φ. |
| `R_OUTER_INIT, R_INNER_INIT, MARGIN` | 50, 12.5, 2.5 | Pre-swell geometry.  Sized so swell compresses to R_outer ≈ 32, R_inner ≈ 8 — matching `N_inner = 8`, `N_width = 12` in the design recipe. |


In [ ]:
P_TARGET     = 635
N_NODES      = 60
PHI_TARGET   = 0.84
OH_DRAG      = 0.15
N_RELAX      = 15_000
SEG_STEPS    = 50_000
N_SEGMENTS   = 12              # ← drop to 2 for a quick runtime test
SAMPLE_DRIVE = 1250
OMEGA        = 0.002
FPS          = 12
E_CAND       = 64
DT_FACTOR_DRIVE = 0.8
POLY_DIST    = {'type': 'bimodal', 'ratio': 0.5, 'delta': 0.2}

# Pre-swell cell sizing (will compress to R_out≈32, R_in≈8 after swell)
R_OUTER_INIT = 50.0
R_INNER_INIT = 12.5
MARGIN       = 2.5

Lx = Ly = 2 * R_OUTER_INIT + 2 * MARGIN
CX = CY = Lx / 2.0

print(f'P={P_TARGET}, N_nodes={N_NODES}, target_phi={PHI_TARGET}')
print(f'Initial cell: R_in={R_INNER_INIT}, R_out={R_OUTER_INIT}, Lx={Lx}')
print(f'Drive: Omega={OMEGA}, dt_factor={DT_FACTOR_DRIVE}, '
       f'{N_SEGMENTS} × {SEG_STEPS} = {N_SEGMENTS * SEG_STEPS} steps')


## 4. Imports and helper functions

- `t1_capture` is the per-frame callback that grabs velocity state + neighbour
  topology.  Called every `SAMPLE_DRIVE` steps by the integrator.
- `save_segment` does an **atomic write**: writes to a `_tmp.npz`, then renames.
  This means a process kill mid-write never corrupts the file.
- `write_meta` similarly atomic; tracks segment counter + ring assignments.
- `load_segments_for_render` re-reads all segment files at the end to build the
  cumulative movie.


In [ ]:
import time, json, shutil
import numpy as np
import tensorflow as tf
import src.simulation.tf_sim as tf_sim_mod
tf_sim_mod.set_dtype(tf.float64)

from src.simulation.tf_sim import make_traj
from src.epd.particles import ParticleSpec
from src.epd.objects import CouetteCell
from src.epd.system import System

STATE_SWELL = OUT_DIR / 'state_post_swell.npz'
STATE_DRIVE = OUT_DIR / 'state_drive.npz'
META        = OUT_DIR / 'meta.npz'
NPZ         = OUT_DIR / 'drive_data.npz'
FULL        = OUT_DIR / 'full.mp4'


def t1_capture(sys_h):
    """Per-frame snapshot: velocity state + neighbour topology (uint16)."""
    st = sys_h._state
    return {
        'v_cm':     st['v_cm'].numpy(),
        'omega':    st['omega'].numpy(),
        'u_dot':    st['u_dot'].numpy(),
        'cap_cand': sys_h._cm_mgr.CapCandidates.astype(np.uint16),
    }


def save_segment(seg_idx_1based, frames, callbacks, path):
    tmp = path.with_name(path.stem + '_tmp.npz')
    np.savez(tmp,
        seg_idx = np.int32(seg_idx_1based),
        t      = np.array([f['t']    for f in frames], dtype=np.float64),
        step   = np.array([f['step'] for f in frames], dtype=np.int64),
        x_all  = np.stack([f['x_all'] for f in frames]),
        theta  = np.stack([f['theta'] for f in frames]),
        v_cm   = np.stack([cb['v_cm']    for cb in callbacks]),
        omega  = np.stack([cb['omega']   for cb in callbacks]),
        u_dot  = np.stack([cb['u_dot']   for cb in callbacks]),
        cap_cand = np.stack([cb['cap_cand'] for cb in callbacks]),
    )
    os.replace(tmp, path)


def write_meta(last_seg, inner_idx, outer_idx):
    tmp = META.with_name(META.stem + '_tmp.npz')
    np.savez(tmp,
        last_seg  = np.int32(last_seg),
        inner_idx = np.array(inner_idx, dtype=np.int32),
        outer_idx = np.array(outer_idx, dtype=np.int32),
    )
    os.replace(tmp, META)


def load_segments_for_render(out_dir, n_segments):
    out = []
    for s in range(1, n_segments + 1):
        f = out_dir / f'segment_{s:03d}.npz'
        if not f.exists():
            break
        d = np.load(f)
        for i in range(d['t'].shape[0]):
            out.append({
                't':     float(d['t'][i]),
                'step':  int(d['step'][i]),
                'x_all': d['x_all'][i],
                'theta': d['theta'][i],
                'x_cm':  d['x_all'][i].mean(axis=1),
            })
    return out


## 5. Resume detection

If we've previously run this notebook (and Drive has the checkpoints), the
resume logic detects them and we skip back to the next-needed phase:

| Detected | Action |
|---|---|
| `state_drive.npz` + `meta.npz` exist, `last_seg < N_SEGMENTS` | Skip phases 1-3, resume phase 4 from `last_seg + 1` |
| `state_post_swell.npz` exists, no drive checkpoint | Skip phase 1 swell; redo phases 2-4 from scratch |
| Nothing exists | Full clean run |

Phase 1 (swell) is not checkpointed mid-loop; if the notebook is killed during
swell, you redo the swell on next run (~15-25 min depending on GPU).  Once swell
completes, every subsequent state is durable across kills.


In [ ]:
resume_seg = 0
resume_inner_idx = None
resume_outer_idx = None

if META.exists() and STATE_DRIVE.exists():
    m = np.load(META, allow_pickle=True)
    resume_seg       = int(m['last_seg'])
    resume_inner_idx = m['inner_idx'].tolist()
    resume_outer_idx = m['outer_idx'].tolist()
    if resume_seg >= N_SEGMENTS:
        print(f'[resume] all {N_SEGMENTS} segments already done; jumping to final render')
    else:
        print(f'[resume] state_drive + meta found; resuming at segment {resume_seg + 1}/{N_SEGMENTS}')
elif STATE_SWELL.exists():
    print(f'[resume] post-swell state found but no drive checkpoint; will re-run phase 2 (relax)')
else:
    print(f'[resume] no checkpoints; running from scratch')


## 6. Build system + Phase 1 — swell to target φ (or restore from checkpoint)

Three branches:
1. **Resuming from drive checkpoint** → minimal init, restore `state_drive.npz`.
2. **Resuming from post-swell** → minimal init, restore `state_post_swell.npz`.
3. **Fresh start** → run the full adaptive swell with conservative parameters
   (`dphi_init=4e-4`, `n_relax=400`, `swell_alpha=15`).  Tuned for φ near RCP
   where the swell can otherwise hit circularity-collapse rollbacks.

Note `dt_factor=0.5` during construction.  This is conservative — set so the swell
phase doesn't trip the closing-rate watchdog.  We bump it back up before the drive.


In [ ]:
# Build the system
sys_ce = System(Lx, Ly, periodic_x=False, periodic_y=False,
                dt_factor=0.5, E_candidates=E_CAND)
cell = CouetteCell(inner_radius=R_INNER_INIT, outer_radius=R_OUTER_INIT,
                   x0=CX, y0=CY)
cell.set_render(color='#333333', linewidth=2.0, alpha=0.9)
sys_ce.add_object(cell)
spec = ParticleSpec(count=P_TARGET, type='emulsion', gamma=1.0, q=5,
                    N_nodes=N_NODES, Oh=OH_DRAG, poly_dist=POLY_DIST)
sys_ce.add_particles(spec)

t0 = time.time()
if STATE_DRIVE.exists() and resume_seg > 0:
    print('[phase 1] skipped — resuming from drive checkpoint')
    sys_ce.initialize(phi_target=PHI_TARGET, seed=42, verbose=False,
                      relax_only=True, n_relax_init=0)
    sys_ce.restore_state(STATE_DRIVE)
elif STATE_SWELL.exists():
    print('[phase 1] STATE_SWELL exists — initializing minimal then restoring')
    sys_ce.initialize(phi_target=PHI_TARGET, seed=42, verbose=False,
                      relax_only=True, n_relax_init=0)
    sys_ce.restore_state(STATE_SWELL)
else:
    print(f'[phase 1] swell to phi={PHI_TARGET} ...')
    sys_ce.initialize(phi_target=PHI_TARGET, seed=42, verbose=True,
                      n_relax=400, swell_alpha=15.0,
                      dphi_init=0.0004, dphi_max=0.004)
    sys_ce.save_state(STATE_SWELL)
    print(f'[phase 1] saved post-swell state → {STATE_SWELL.name}')

print(f'[phase 1] {time.time()-t0:.1f}s  '
       f'R_in={cell._inner_r:.3f}  R_out={cell._outer_r:.3f}  '
       f'phi={sys_ce.phi_outer:.4f}  Lx={sys_ce.Lx:.3f}')


## 7. Phase 2 — equilibrating relax

15 000 steps with drag (`Oh=0.15`) and no driving.  Drains the kinetic energy
left over from the swell so the driven phase starts from mechanical rest
(`ρ_max → 0`).  At dt_factor=1.5 this is fast (no driving = no closing-rate risk).

Skipped if resuming from a drive checkpoint (already past this).


In [ ]:
if resume_seg == 0:
    sys_ce.dt_factor = 1.5
    print(f'[phase 2] relax {N_RELAX} steps ...')
    t1 = time.time()
    sys_ce.run(N_RELAX, sample_every=N_RELAX, verbose=True)
    sys_ce.clear_recording()
    print(f'[phase 2] {time.time()-t1:.1f}s  rho_max={sys_ce._max_disp_ratio_run:.4f}  '
           f'phi_outer={sys_ce.phi_outer:.4f}')
else:
    print('[phase 2] skipped — resuming from drive checkpoint')


## 8. Phase 3 — identify driven inner ring + pinned outer ring

**Inner ring** (rotating):  droplets within `R_inner + 1.5·R0` of the cell
centre.  These get a `make_traj` with both orbital and body angular velocity
= Ω, so they orbit the cell centre while their bodies co-rotate (rough side
stays tangent to the inner wall).

**Outer ring** (pinned):  droplets within `2·R0` of the outer wall.  These
get a zero-trajectory `make_traj` (frozen in place) — required because the
outer wall is frictionless and would otherwise let the bulk co-rotate rigidly.

Both rings are checked for full angular coverage; max gap > 90° would mean a
hole in the rough boundary.


In [ ]:
cx, cy = cell._origin[0], cell._origin[1]
R0_mean = float(np.mean([p.R0 for p in sys_ce._particles]))

if resume_seg > 0:
    inner_idx = resume_inner_idx
    outer_idx = resume_outer_idx
    print(f'[phase 3] resumed: inner={len(inner_idx)}, outer={len(outer_idx)}')
else:
    x_cm = sys_ce._state['x_cm'].numpy()
    r_from_center = np.sqrt((x_cm[:,0] - cx)**2 + (x_cm[:,1] - cy)**2)
    inner_thresh  = cell._inner_r + 1.5 * R0_mean
    outer_thresh  = cell._outer_r - 2.0 * R0_mean
    inner_idx = [i for i in range(P_TARGET) if r_from_center[i] < inner_thresh]
    outer_idx = [i for i in range(P_TARGET) if r_from_center[i] > outer_thresh]

    def _ring_gap(idx):
        if len(idx) < 2: return 360.0
        a = np.sort(np.degrees(np.arctan2(x_cm[idx,1] - cy,
                                           x_cm[idx,0] - cx)) % 360)
        return float(np.max(np.concatenate([np.diff(a),
                                             [360 - a[-1] + a[0]]])))

    print(f'[phase 3] inner ring: {len(inner_idx)} droplets (gap {_ring_gap(inner_idx):.1f}°)')
    print(f'[phase 3] outer ring: {len(outer_idx)} droplets (gap {_ring_gap(outer_idx):.1f}°)')

# Build trajectories: inner = orbital + body co-rotation, outer = pinned
inner_traj = make_traj(omega_orbit_dc=OMEGA, omega_spin_dc=OMEGA, r_ref=(cx, cy))
zero_traj  = make_traj()  # all zeros — pinned

driven_idx = list(inner_idx) + list(outer_idx)
traj_rows  = ([inner_traj] * len(inner_idx)) + ([zero_traj] * len(outer_idx))
sys_ce.set_driven_particles(driven_idx, traj_rows, frozen=True)

for i in inner_idx:
    sys_ce._particle_colors[i] = '#e05252'  # red
for i in outer_idx:
    sys_ce._particle_colors[i] = '#4477aa'  # blue


## 9. Phase 4 — driven shear in 12 segments of 50 000 steps

Each segment:
1. Run `SEG_STEPS = 50 000` integration steps with `t1_capture` callback firing
   every `SAMPLE_DRIVE = 1250` steps (40 captures per segment).
2. **Atomically** save to Drive: `segment_NNN.npz` (this segment's frames + cap_cand),
   `state_drive.npz` (current state), `meta.npz` (last_seg + ring assignments).
3. Render a quick preview mp4 of just this segment's 40 frames.
4. Free this segment's data from in-memory `frames` / `callback_data` lists
   (already on disk) so memory stays bounded.

If the kernel dies mid-way, re-run the notebook — the resume logic in cell 6
will detect the partially-completed segment count and continue.

**Watchdog monitoring:**  closing-rate ρ_contact is reported each segment.
Advisory at 0.05, strong at 0.10, critical at 0.20.  At our `dt_factor=0.8`
and `Ω=0.002` we expect ρ_max ≲ 0.01 throughout (5× under advisory).


In [ ]:
sys_ce.dt_factor = DT_FACTOR_DRIVE
print(f'[phase 4] {N_SEGMENTS}×{SEG_STEPS} steps  Omega={OMEGA}  '
       f'dt_factor={DT_FACTOR_DRIVE}  start_seg={resume_seg}')
t2 = time.time()

for seg in range(resume_seg, N_SEGMENTS):
    pre_frames = len(sys_ce.frames)
    pre_cb     = len(sys_ce.callback_data)
    t_seg = time.time()
    sys_ce.run(SEG_STEPS, sample_every=SAMPLE_DRIVE,
               callback=t1_capture, verbose=False,
               record_initial=(seg == 0 and resume_seg == 0))
    n_new       = len(sys_ce.frames) - pre_frames
    new_frames  = sys_ce.frames[pre_frames:]
    new_callbs  = sys_ce.callback_data[pre_cb:]
    cumul_step  = (seg + 1) * SEG_STEPS

    # Atomic checkpoint
    seg_path = OUT_DIR / f'segment_{seg+1:03d}.npz'
    save_segment(seg + 1, new_frames, new_callbs, seg_path)
    sys_ce.save_state(STATE_DRIVE)
    write_meta(seg + 1, inner_idx, outer_idx)

    # Free this segment's data from memory
    sys_ce.frames        = sys_ce.frames[:pre_frames]
    sys_ce.callback_data = sys_ce.callback_data[:pre_cb]

    print(f'  segment {seg+1}/{N_SEGMENTS}  step={cumul_step}  '
          f'sim_t={sys_ce.t:.2f}  +{n_new} frames  ({time.time()-t_seg:.0f}s)  '
          f'rho_max={sys_ce._max_disp_ratio_run:.4f}  -> {seg_path.name}')

    # Per-segment preview
    preview = OUT_DIR / f'preview_step{cumul_step//1000:03d}k.mp4'
    sys_ce.make_movie(
        preview, fps=FPS, n_arc=4,
        xlim=(-0.5, sys_ce.Lx + 0.5),
        ylim=(-0.5, sys_ce.Ly + 0.5),
        title=(f'T1 bidisperse — step {cumul_step}/{N_SEGMENTS*SEG_STEPS}  '
                f'phi={sys_ce.phi_outer:.3f}  Omega={OMEGA}'),
        frames=new_frames,
    )
    print(f'  preview → {preview.name}')

print(f'[phase 4] {time.time()-t2:.0f}s total drive (this run)')


## 10. Final — concatenate all segments, save full T1 dataset, render cumulative movie

After all segments complete, the per-segment `.npz` files are concatenated into
a single `drive_data.npz` for offline post-processing.  The cumulative movie
stitches all 480 frames into one continuous animation.

**Post-processing pipeline (separate notebook):**  load `drive_data.npz`,
compute Delaunay triangulation per frame, detect T1 events as edge swaps,
recompute per-pair contact forces from positions + neighbours, build per-particle
stress tensors, plot stress relaxation around each event.


In [ ]:
# Concatenate per-segment files
print(f'[save] concatenating {N_SEGMENTS} segments → {NPZ.name} ...')
t3 = time.time()
import glob
seg_files = sorted(OUT_DIR.glob('segment_*.npz'))
data = {k: [] for k in ('t','step','x_all','theta','v_cm','omega','u_dot','cap_cand')}
for sf in seg_files:
    d = np.load(sf)
    for k in data: data[k].append(d[k])
data = {k: np.concatenate(v, axis=0) for k, v in data.items()}

# Per-particle / per-system params (immutable through the run)
_p = sys_ce._params
def _np(k):
    v = _p.get(k)
    if v is None: return None
    return v.numpy() if hasattr(v, 'numpy') else np.asarray(v)

np.savez(NPZ,
    **data,
    P            = np.int32(P_TARGET),
    N_nodes      = np.int32(N_NODES),
    E_candidates = np.int32(E_CAND),
    Lx           = np.float64(sys_ce.Lx),
    Ly           = np.float64(sys_ce.Ly),
    Omega        = np.float64(OMEGA),
    dt           = np.float64(sys_ce.dt),
    inner_idx    = np.array(inner_idx, dtype=np.int32),
    outer_idx    = np.array(outer_idx, dtype=np.int32),
    cell_origin  = cell._origin.astype(np.float64),
    cell_R_inner = np.float64(cell._inner_r),
    cell_R_outer = np.float64(cell._outer_r),
    r_c_per_p        = _np('r_c_per_p'),
    k_c_per_p        = _np('k_c_per_p'),
    El_t_per_p       = _np('El_t_per_p'),
    K_area_per_p     = _np('K_area_per_p'),
    EI_per_p         = _np('EI_per_p'),
    gamma_per_p      = _np('gamma_per_p'),
    xi_drag_per_p    = _np('xi_drag_per_p'),
    m_node_per_p     = _np('m_node_per_p'),
    M_disk_per_p     = _np('M_disk_per_p'),
    I_disk_per_p     = _np('I_disk_per_p'),
    alpha_damp_per_p = _np('alpha_damp_per_p'),
    L0_per_p         = _np('L0'),
    R0_per_p         = np.array([p.R0 for p in sys_ce._particles], dtype=np.float64),
    X_ref            = sys_ce._state['X_ref'].numpy(),
)
print(f'[save] {time.time()-t3:.1f}s -> {NPZ}  ({NPZ.stat().st_size / 1e9:.2f} GB)')

# Cumulative movie
print(f'\n[render] cumulative movie from segment files ...')
t4 = time.time()
sys_ce.frames = load_segments_for_render(OUT_DIR, N_SEGMENTS)
sys_ce.make_movie(
    FULL, fps=FPS, n_arc=4,
    xlim=(-0.5, sys_ce.Lx + 0.5),
    ylim=(-0.5, sys_ce.Ly + 0.5),
    title=(f'T1 bidisperse — phi={sys_ce.phi_outer:.3f}  Omega={OMEGA}  '
            f'P={P_TARGET}  driven={len(inner_idx)}  pinned={len(outer_idx)}'),
)
print(f'[render] {time.time()-t4:.0f}s -> {FULL}')
print(f'\n[total] {(time.time()-t0)/60:.1f} min')
